# IMDb Movie Recommendation System

## 2. NLP Preprocessing

This notebook prepares movie titles and storylines for the Natural Language Processing (NLP) stage of the IMDb Movie Recommendation System.

The recommendation engine will compare a user's input storyline with movie storylines using TF-IDF and cosine similarity. Therefore, the textual data must be normalized into a consistent representation before vectorization.

### Preprocessing Objectives

- Normalize movie titles and storylines.
- Remove unnecessary characters and punctuation.
- Remove the numerical indexing present in movie titles.
- Tokenize storyline text.
- Remove common English stopwords.
- Preserve meaningful words for semantic similarity.
- Save the processed dataset for the recommendation stage.

## Preprocessing Strategy

The movie storylines are textual data, so they must be normalized before they can be converted into numerical representations.

The preprocessing pipeline used in this project is:

1. Load the original dataset.
2. Preserve the raw storyline for comparison and validation.
3. Remove the numerical prefix from movie titles.
4. Convert storyline text to lowercase.
5. Remove punctuation and non-alphabetic characters.
6. Normalize extra whitespace.
7. Tokenize the storyline into individual words.
8. Remove common English stopwords.
9. Reconstruct the processed storyline as a clean text string.
10. Validate the processed data.
11. Save the processed dataset for the recommendation stage.

The processed storyline will later be transformed into TF-IDF vectors for similarity-based movie recommendation.

## Why These Preprocessing Steps?

### Lowercasing
Converts words such as `Movie`, `movie`, and `MOVIE` into the same representation.

### Punctuation Removal
Punctuation generally does not contribute useful information to the TF-IDF representation used in this project.

### Whitespace Normalization
Prevents unnecessary spaces from creating inconsistent text representations.

### Tokenization
Splits a storyline into individual words so that text can be processed at the word level.

### Stopword Removal
Removes very common words such as `the`, `a`, `is`, and `of` that generally contribute little to distinguishing one storyline from another.

### Title Number Removal
The dataset contains numerical indexing such as `1. The Fall Guy`. The number identifies the row rather than the movie, so it is removed from the movie title.

In [1]:
from pathlib import Path
import sys

import pandas as pd
import re


# Locate project root dynamically
current_dir = Path.cwd()

project_root = next(
    (
        path
        for path in [current_dir, *current_dir.parents]
        if (path / "src" / "config.py").exists()
    ),
    None
)

if project_root is None:
    raise FileNotFoundError(
        "Project root could not be located."
    )

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

In [2]:
from src.config import RAW_DATA_PATH, DATA_DIR

print("Project root:", project_root)
print("Raw dataset:", RAW_DATA_PATH)

Project root: d:\Projects\Python Projects\Python Mini Projects\IMDB_Movie_Recommendation_System
Raw dataset: D:\Projects\Python Projects\Python Mini Projects\IMDB_Movie_Recommendation_System\data\IMDBRecSys.csv


In [3]:
df = pd.read_csv(RAW_DATA_PATH)

df.head()

,Movie_Title,Storyline
0,1. The Fall Guy,"A stuntman, fresh off an almost career-ending ..."
1,2. The Substance,A fading celebrity takes a black-market drug: ...
2,3. The Life of Chuck,"A life-affirming, genre-bending story about th..."
3,4. Abigail,After a group of criminals kidnap the ballerin...
4,5. The Ministry of Ungentlemanly Warfare,The British military recruits a small group of...


In [4]:
df.shape

(6021, 2)

In [5]:
processed_df = df.copy()

In [6]:
processed_df["Movie_Title"] = (
    processed_df["Movie_Title"]
    .str.replace(r"^\s*\d+\.\s*", "", regex=True)
    .str.strip()
)

In [7]:
processed_df["Movie_Title"].head(10)

0                             The Fall Guy
1                            The Substance
2                        The Life of Chuck
3                                  Abigail
4    The Ministry of Ungentlemanly Warfare
5                                    Relay
6                           Dune: Part Two
7                                    Anora
8                         Jab Khuli Kitaab
9                           I'm Still Here
Name: Movie_Title, dtype: object

In [8]:
processed_df["Storyline_Raw"] = processed_df["Storyline"]

In [9]:
def normalize_text(text):
    text = str(text).lower()
    text = re.sub(r"[^a-z\s]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

In [10]:
processed_df["Storyline_Clean"] = (
    processed_df["Storyline"]
    .apply(normalize_text)
)

In [11]:
processed_df[
    ["Movie_Title", "Storyline_Raw", "Storyline_Clean"]
].head()

,Movie_Title,Storyline_Raw,Storyline_Clean
0,The Fall Guy,"A stuntman, fresh off an almost career-ending ...",a stuntman fresh off an almost career ending a...
1,The Substance,A fading celebrity takes a black-market drug: ...,a fading celebrity takes a black market drug a...
2,The Life of Chuck,"A life-affirming, genre-bending story about th...",a life affirming genre bending story about thr...
3,Abigail,After a group of criminals kidnap the ballerin...,after a group of criminals kidnap the ballerin...
4,The Ministry of Ungentlemanly Warfare,The British military recruits a small group of...,the british military recruits a small group of...


In [12]:
processed_df["Tokens"] = (
    processed_df["Storyline_Clean"]
    .str.split()
)

In [13]:
processed_df[
    ["Movie_Title", "Tokens"]
].head()

,Movie_Title,Tokens
0,The Fall Guy,"[a, stuntman, fresh, off, an, almost, career, ..."
1,The Substance,"[a, fading, celebrity, takes, a, black, market..."
2,The Life of Chuck,"[a, life, affirming, genre, bending, story, ab..."
3,Abigail,"[after, a, group, of, criminals, kidnap, the, ..."
4,The Ministry of Ungentlemanly Warfare,"[the, british, military, recruits, a, small, g..."


In [14]:
import nltk

nltk.download("stopwords")

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\admin\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [15]:
from nltk.corpus import stopwords

stop_words = set(stopwords.words("english"))

len(stop_words)

198

In [16]:
processed_df["Tokens"] = processed_df["Tokens"].apply(
    lambda tokens: [
        word
        for word in tokens
        if word not in stop_words
    ]
)

In [17]:
processed_df[
    ["Movie_Title", "Tokens"]
].head()

,Movie_Title,Tokens
0,The Fall Guy,"[stuntman, fresh, almost, career, ending, acci..."
1,The Substance,"[fading, celebrity, takes, black, market, drug..."
2,The Life of Chuck,"[life, affirming, genre, bending, story, three..."
3,Abigail,"[group, criminals, kidnap, ballerina, daughter..."
4,The Ministry of Ungentlemanly Warfare,"[british, military, recruits, small, group, hi..."


In [18]:
processed_df["Storyline_Processed"] = (
    processed_df["Tokens"]
    .apply(" ".join)
)

In [19]:
processed_df[
    [
        "Movie_Title",
        "Storyline_Raw",
        "Storyline_Processed"
    ]
].head()

,Movie_Title,Storyline_Raw,Storyline_Processed
0,The Fall Guy,"A stuntman, fresh off an almost career-ending ...",stuntman fresh almost career ending accident t...
1,The Substance,A fading celebrity takes a black-market drug: ...,fading celebrity takes black market drug cell ...
2,The Life of Chuck,"A life-affirming, genre-bending story about th...",life affirming genre bending story three chapt...
3,Abigail,After a group of criminals kidnap the ballerin...,group criminals kidnap ballerina daughter powe...
4,The Ministry of Ungentlemanly Warfare,The British military recruits a small group of...,british military recruits small group highly s...


In [20]:
print("Rows before preprocessing :", len(df))
print("Rows after preprocessing  :", len(processed_df))

Rows before preprocessing : 6021
Rows after preprocessing  : 6021


In [21]:
print(
    "Empty processed storylines:",
    processed_df["Storyline_Processed"].str.strip().eq("").sum()
)

Empty processed storylines: 0


In [22]:
processed_df[
    processed_df["Storyline_Processed"].str.strip().eq("")
][
    ["Movie_Title", "Storyline_Raw"]
]

,Movie_Title,Storyline_Raw


In [23]:
comparison = processed_df[
    [
        "Movie_Title",
        "Storyline_Raw",
        "Storyline_Processed"
    ]
].head(10)

comparison

,Movie_Title,Storyline_Raw,Storyline_Processed
0,The Fall Guy,"A stuntman, fresh off an almost career-ending ...",stuntman fresh almost career ending accident t...
1,The Substance,A fading celebrity takes a black-market drug: ...,fading celebrity takes black market drug cell ...
2,The Life of Chuck,"A life-affirming, genre-bending story about th...",life affirming genre bending story three chapt...
3,Abigail,After a group of criminals kidnap the ballerin...,group criminals kidnap ballerina daughter powe...
4,The Ministry of Ungentlemanly Warfare,The British military recruits a small group of...,british military recruits small group highly s...
5,Relay,A broker of lucrative payoffs between corrupt ...,broker lucrative payoffs corrupt corporations ...
6,Dune: Part Two,Paul Atreides unites with the Fremen while on ...,paul atreides unites fremen warpath revenge co...
7,Anora,A young stripper from Brooklyn meets and impul...,young stripper brooklyn meets impulsively marr...
8,Jab Khuli Kitaab,Gopal and Anusuya's decades-long marriage face...,gopal anusuya decades long marriage faces uphe...
9,I'm Still Here,A woman married to a former politician during ...,woman married former politician military dicta...


In [24]:
processed_df["raw_word_count"] = (
    processed_df["Storyline_Raw"]
    .str.split()
    .str.len()
)

processed_df["processed_word_count"] = (
    processed_df["Storyline_Processed"]
    .str.split()
    .str.len()
)

processed_df["words_removed"] = (
    processed_df["raw_word_count"]
    - processed_df["processed_word_count"]
)

In [25]:
processed_df[
    [
        "raw_word_count",
        "processed_word_count",
        "words_removed"
    ]
].describe()

,raw_word_count,processed_word_count,words_removed
count,6021.000000,6021.000000,6021.000000
mean,32.730277,19.458894,13.271383
std,20.116917,11.506311,9.386749
min,4.000000,3.000000,0.000000
25%,24.000000,14.000000,8.000000
50%,30.000000,18.000000,12.000000
75%,37.000000,22.000000,16.000000
max,463.000000,267.000000,196.000000


In [26]:
print(
    f"Average raw words       : "
    f"{processed_df['raw_word_count'].mean():.2f}"
)

print(
    f"Average processed words : "
    f"{processed_df['processed_word_count'].mean():.2f}"
)

print(
    f"Average words removed   : "
    f"{processed_df['words_removed'].mean():.2f}"
)

Average raw words       : 32.73
Average processed words : 19.46
Average words removed   : 13.27


In [27]:
sample_text = processed_df.loc[0, "Storyline_Processed"]

remaining_stopwords = [
    word
    for word in sample_text.split()
    if word in stop_words
]

remaining_stopwords

[]

### Tokenization Note

The current implementation uses whitespace-based tokenization with Python's `str.split()` because the dataset contains relatively clean English storyline text.

A dedicated NLP tokenizer can be introduced later if the project requires more sophisticated linguistic processing.

In [28]:
PROCESSED_DATA_PATH = DATA_DIR / "IMDBRecSys_Processed.csv"

In [29]:
processed_df

,Movie_Title,Storyline,Storyline_Raw,Storyline_Clean,Tokens,Storyline_Processed,raw_word_count,processed_word_count,words_removed
0,The Fall Guy,"A stuntman, fresh off an almost career-ending ...","A stuntman, fresh off an almost career-ending ...",a stuntman fresh off an almost career ending a...,"[stuntman, fresh, almost, career, ending, acci...",stuntman fresh almost career ending accident t...,35,20,15
1,The Substance,A fading celebrity takes a black-market drug: ...,A fading celebrity takes a black-market drug: ...,a fading celebrity takes a black market drug a...,"[fading, celebrity, takes, black, market, drug...",fading celebrity takes black market drug cell ...,20,14,6
2,The Life of Chuck,"A life-affirming, genre-bending story about th...","A life-affirming, genre-bending story about th...",a life affirming genre bending story about thr...,"[life, affirming, genre, bending, story, three...",life affirming genre bending story three chapt...,17,13,4
3,Abigail,After a group of criminals kidnap the ballerin...,After a group of criminals kidnap the ballerin...,after a group of criminals kidnap the ballerin...,"[group, criminals, kidnap, ballerina, daughter...",group criminals kidnap ballerina daughter powe...,30,17,13
4,The Ministry of Ungentlemanly Warfare,The British military recruits a small group of...,The British military recruits a small group of...,the british military recruits a small group of...,"[british, military, recruits, small, group, hi...",british military recruits small group highly s...,23,17,6
...,...,...,...,...,...,...,...,...,...
6016,The Lamb,A gripping story about a man who was the terro...,A gripping story about a man who was the terro...,a gripping story about a man who was the terro...,"[gripping, story, man, terror, russian, empire...",gripping story man terror russian empire legen...,28,15,13
6017,Johatsu,A barber in a Roman neighborhood believes in o...,A barber in a Roman neighborhood believes in o...,a barber in a roman neighborhood believes in o...,"[barber, roman, neighborhood, believes, online...",barber roman neighborhood believes online cons...,29,19,10
6018,Societat negra,"Follows David Edison, a cam-girl obsessed I.T....","Follows David Edison, a cam-girl obsessed I.T....",follows david edison a cam girl obsessed i t g...,"[follows, david, edison, cam, girl, obsessed, ...",follows david edison cam girl obsessed guy bec...,27,19,8
6019,El Ascenso,Simon Says is a thriller about a traumatized (...,Simon Says is a thriller about a traumatized (...,simon says is a thriller about a traumatized p...,"[simon, says, thriller, traumatized, ptsd, col...",simon says thriller traumatized ptsd college g...,30,19,11


In [30]:
processed_df[
    [
        "Movie_Title",
        "Storyline_Raw",
        "Storyline_Processed"
    ]
].to_csv(
    PROCESSED_DATA_PATH,
    index=False
)

In [31]:
print("Processed dataset saved to:")
print(PROCESSED_DATA_PATH)

Processed dataset saved to:
D:\Projects\Python Projects\Python Mini Projects\IMDB_Movie_Recommendation_System\data\IMDBRecSys_Processed.csv
